In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd, numpy as np

df = pd.read_csv('cleaned_data.csv')
# model = SentenceTransformer('all-MiniLM-L6-v2')
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

embeddings = model.encode(df['description_clean'].tolist(), show_progress_bar=True)
np.save('club_embeddings.npy', embeddings)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend(user_text, top_n=5, category_filter=None):
    query_embedding = model.encode([user_text])
    scores = cosine_similarity(query_embedding, embeddings)[0]
    df['score'] = scores

    result = df.copy()
    if category_filter:
        result = result[result['category'].isin(category_filter)]

    return result.nlargest(top_n, 'score')[['name','category','description_clean','score','telegram_url','instagram_url']]

In [ ]:
def normalize_query(user_text):


In [ ]:
# USER_QUERY = ""
# USER_QUERY = normalize_query(USER_QUERY)
recommend("i like programming and wanna participate in hackathons")

^^^ not working in kazakh

In [ ]:
print(df[df['name'] == "NU Shyraq"])

In [ ]:
# # test_similarity.py — run this today
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity
# import pandas as pd, numpy as np

# df = pd.read_csv('updated_clubs.csv')
# model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# embeddings = model.encode(df['description'].tolist())

# # test with raw input first
test_queries = [
    "я люблю футбол и хочу новых друзей",
    "i like programming and participating in hackathons",
    "I like photography and creative arts",
]

for q in test_queries:
    vec = model.encode([q])
    scores = cosine_similarity(vec, embeddings)[0]
    df['score'] = scores
    top3 = df.nlargest(3, 'score')[['name','category','score']]
    print(f"\nQuery: {q}")
    print(top3.to_string())